In [ ]:
import sys
import torch

print("Python:", sys.version)
print("Executable:", sys.executable)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

In [1]:
import transformers
import datasets
import trl
import peft
import accelerate

print("All libraries loaded successfully!")

All libraries loaded successfully!


In [2]:
import torch
import json
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

In [3]:
import json
from datasets import Dataset

def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

train_data = load_jsonl("train_chat.jsonl")
test_data = load_jsonl("test_chat.jsonl")

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

print(train_dataset[0])

{'messages': [{'content': "Flygon ex δ\nHP 150\nType Psychic\nAbility Sand Damage: As long as Flygon ex is your Active Pokémon, put 1 damage counter on each of your opponent's Benched Basic Pokémon between turns. You can't use more than 1 Sand Damage Poké-Body between turns.\nRetreat Colorless, Colorless\nAttack Psychic Pulse, 80: Does 10 damage to each of your opponent's Benched Pokémon that has any damage counters on it. (Don't apply Weakness and Resistance for Benched Pokémon.)", 'role': 'user'}, {'content': '{"name": "Flygon ex δ", "hp": "150", "types": ["Psychic"], "abilities": [{"name": "Sand Damage", "text": "As long as Flygon ex is your Active Pokémon, put 1 damage counter on each of your opponent\'s Benched Basic Pokémon between turns. You can\'t use more than 1 Sand Damage Poké-Body between turns."}], "attacks": [{"name": "Psychic Pulse", "damage": "80", "text": "Does 10 damage to each of your opponent\'s Benched Pokémon that has any damage counters on it. (Don\'t apply Weakn

In [4]:
print(len(train_data))

3833


In [5]:
#only baseline model use it
BASELINE_PROMPT = """
Extract the key information from the following Pokémon card text and return ONLY valid JSON.

Required keys:
name, hp, types, abilities, attacks, weaknesses, retreatCost

Rules:
- Return JSON only
- Do not include explanation
- abilities must be a list of objects with keys: name, text
- attacks must be a list of objects with keys: name, damage, text
- weaknesses must be a list of objects with keys: type, value
- retreatCost must be a list of strings

Card text:
{card_text}
"""

In [6]:
#baseline generation
def generate_baseline_response(model, tokenizer, user_text, max_new_tokens=300):
    prompt = BASELINE_PROMPT.format(card_text=user_text)

    messages = [
        {
            "role": "system",
            "content": "You are an information extraction assistant that returns JSON only."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

In [7]:
import json

def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

test_dataset = load_jsonl("test_dataset.jsonl")

fixed_test_inputs = [x["input"] for x in test_dataset[:100]]
fixed_ground_truths = [x["output"] for x in test_dataset[:100]]

In [8]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

adapter_path = "./qwen_ptcg_lora_adapter"   

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

finetuned_model = PeftModel.from_pretrained(base_model, adapter_path)
finetuned_model.eval()

print("Fine-tuned model loaded")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Fine-tuned model loaded


In [9]:
baseline_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
baseline_model.eval()

print("Baseline model loaded")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Baseline model loaded


In [10]:
import json

def try_parse_json(text):
    try:
        return json.loads(text), True
    except:
        pass

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(text[start:end + 1]), True
        except:
            pass

    return None, False

In [11]:
baseline_eval_results = []

for input_text, gold in zip(fixed_test_inputs, fixed_ground_truths):
    raw_output = generate_baseline_response(
        baseline_model,
        tokenizer,
        input_text,
        max_new_tokens=300
    )

    pred_json, ok = try_parse_json(raw_output)

    baseline_eval_results.append({
        "input": input_text,
        "ground_truth": gold,
        "prediction_raw": raw_output,
        "parsed_output": pred_json,
        "valid_json": ok
    })

print("Baseline done:", len(baseline_eval_results))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Baseline done: 100


In [12]:
def generate_json_response(model, tokenizer, user_text, max_new_tokens=300):
    messages = [
        {"role": "user", "content": user_text}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

In [13]:
finetuned_results = []

for input_text, gold in zip(fixed_test_inputs, fixed_ground_truths):
    raw_output = generate_json_response(
        finetuned_model,
        tokenizer,
        input_text,
        max_new_tokens=300
    )

    pred_json, ok = try_parse_json(raw_output)

    finetuned_results.append({
        "input": input_text,
        "ground_truth": gold,
        "prediction_raw": raw_output,
        "parsed_output": pred_json,
        "valid_json": ok
    })

print("Fine-tuned done:", len(finetuned_results))

Fine-tuned done: 100


In [5]:
#Use for model training
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [6]:
# Add LoRA config
from peft import LoraConfig

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [7]:
from transformers import AutoTokenizer

base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

In [8]:
# Create the trainer
from datasets import Dataset

small_test_dataset = test_dataset.select(range(100))
small_train_dataset = train_dataset.select(range(1000))

from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./qwen_ptcg_lora",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=10,
    eval_steps=50,
    save_steps=50,
    max_length=1024,
    packing=False,
    fp16=True,
    dataloader_pin_memory=True,
    eval_strategy="steps"
)

trainer = SFTTrainer(
    model=base_model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_test_dataset,
    processing_class=tokenizer,
    peft_config=peft_config
)

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

In [9]:
trainer.train()
trainer.model.save_pretrained("./qwen_ptcg_lora_adapter")
tokenizer.save_pretrained("./qwen_ptcg_lora_adapter")

print("Adapter saved!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,0.972502,0.883839
100,0.562118,0.620609
150,0.540680,0.539285
200,0.511769,0.509952
250,0.516629,0.488947
300,0.502791,0.479182
350,0.510632,0.463679
400,0.397657,0.459579
450,0.463725,0.449561
500,0.412697,0.441865


Adapter saved!


In [14]:
baseline_inputs = [x["input"] for x in baseline_eval_results]
finetuned_inputs = [x["input"] for x in finetuned_results]

print(baseline_inputs == finetuned_inputs)

True


In [15]:
FIELDS = ["name", "hp", "types", "abilities", "attacks", "weaknesses", "retreatCost"]

def normalize(v):
    if isinstance(v, dict):
        return {k: normalize(v[k]) for k in sorted(v)}
    if isinstance(v, list):
        return [normalize(x) for x in v]
    if isinstance(v, (int, float)):
        return str(v)
    return v

def compute_metrics(results):
    valid_json = 0
    exact_match = 0
    field_correct = {f: 0 for f in FIELDS}

    for item in results:
        pred = item["parsed_output"]
        gold = item["ground_truth"]

        if item["valid_json"] and pred is not None:
            valid_json += 1

            pred_norm = normalize(pred)
            gold_norm = normalize(gold)

            if pred_norm == gold_norm:
                exact_match += 1

            for f in FIELDS:
                if pred_norm.get(f) == gold_norm.get(f):
                    field_correct[f] += 1

    n = len(results)

    metrics = {
        "total_samples": n,
        "valid_json_rate": valid_json / n,
        "exact_match_accuracy": exact_match / n,
    }

    for f in FIELDS:
        metrics[f"{f}_accuracy"] = field_correct[f] / n

    return metrics

baseline_metrics = compute_metrics(baseline_eval_results)
finetuned_metrics = compute_metrics(finetuned_results)

print("Baseline:")
print(json.dumps(baseline_metrics, indent=2))

print("\nFine-tuned:")
print(json.dumps(finetuned_metrics, indent=2))

Baseline:
{
  "total_samples": 100,
  "valid_json_rate": 0.92,
  "exact_match_accuracy": 0.13,
  "name_accuracy": 0.83,
  "hp_accuracy": 0.92,
  "types_accuracy": 0.92,
  "abilities_accuracy": 0.53,
  "attacks_accuracy": 0.38,
  "weaknesses_accuracy": 0.5,
  "retreatCost_accuracy": 0.37
}

Fine-tuned:
{
  "total_samples": 100,
  "valid_json_rate": 1.0,
  "exact_match_accuracy": 0.61,
  "name_accuracy": 0.99,
  "hp_accuracy": 1.0,
  "types_accuracy": 1.0,
  "abilities_accuracy": 0.99,
  "attacks_accuracy": 0.91,
  "weaknesses_accuracy": 0.73,
  "retreatCost_accuracy": 0.77
}


In [16]:
train_names = set()
for x in train_data:
    train_names.add(json.loads(x["messages"][1]["content"])["name"])

test_names = set()
for x in test_data:
    test_names.add(json.loads(x["messages"][1]["content"])["name"])

overlap = train_names.intersection(test_names)

print("overlap count:", len(overlap))
print(list(overlap)[:20])

overlap count: 380
['Pinsir', 'Simisage', 'Whirlipede', 'Shiinotic', 'Unown [C]', 'Salamence', 'Flygon ex', 'Hoppip', 'Archen', 'Heracross', 'Golbat', 'Manaphy', 'Parasect', 'Suicune', 'Unown [O]', 'Spinda', 'Typhlosion ex', 'Smeargle', 'Kyogre ex', 'Eevee']


In [17]:
#Save raw outputs for report/debugging
def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

save_jsonl(baseline_eval_results, "baseline_predictions.jsonl")
save_jsonl(finetuned_results, "finetuned_predictions.jsonl")

In [18]:
import json

train_outputs = set(x["messages"][1]["content"] for x in train_data)
test_outputs = set(x["messages"][1]["content"] for x in test_data)

overlap_outputs = train_outputs & test_outputs
print("exact output overlap:", len(overlap_outputs))

exact output overlap: 0


In [19]:
for i in range(3):
    print("=" * 80)
    print(f"SAMPLE {i}")

    print("\nINPUT:")
    print(finetuned_results[i]["input"])

    print("\nGROUND TRUTH:")
    print(finetuned_results[i]["ground_truth"])

    print("\nBASELINE RAW OUTPUT:")
    print(baseline_eval_results[i]["prediction_raw"])

    print("\nFINE-TUNED RAW OUTPUT:")
    print(finetuned_results[i]["prediction_raw"])

SAMPLE 0

INPUT:
Palkia & Dialga LEGEND
HP 160
Type Water, Metal
Weak Lightning ×2; Fire ×2
Retreat Colorless, Colorless, Colorless
Attack Sudden Delete, : Choose 1 of your opponent's Benched Pokémon. Put that Pokémon and all cards attached to it back to your opponent's hand.
Attack Time Control, : Discard all Metal Energy attached to Palkia & Dialga LEGEND. Add the top 2 cards of your opponent's deck to his or her Prize cards.

GROUND TRUTH:
{'name': 'Palkia & Dialga LEGEND', 'hp': '160', 'types': ['Water', 'Metal'], 'abilities': [], 'attacks': [{'name': 'Sudden Delete', 'damage': '', 'text': "Choose 1 of your opponent's Benched Pokémon. Put that Pokémon and all cards attached to it back to your opponent's hand."}, {'name': 'Time Control', 'damage': '', 'text': "Discard all Metal Energy attached to Palkia & Dialga LEGEND. Add the top 2 cards of your opponent's deck to his or her Prize cards."}], 'weaknesses': [{'type': 'Lightning', 'value': '×2'}, {'type': 'Fire', 'value': '×2'}], 're